In [ ]:
import openai
print(openai.__version__)

In [ ]:
! pip install crewai==1.10.1 faiss-cpu gitpython==3.1.46 langchain-text-splitters

In [6]:
import crewai
print(crewai.__version__)

1.10.1


In [7]:
import git
import os

def clone_repo(repo_url):
    
    repo_path = "code_analisis_repo"

    if os.path.exists(repo_path):
        return repo_path

    git.Repo.clone_from(repo_url, repo_path)
    return repo_path

repo = clone_repo("https://github.com/Ahmed2797/Travel-Planner-Agent.git")

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document 
from langchain_classic.document_loaders import DirectoryLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [19]:
def build_vector_db(repo_path):

    loader = DirectoryLoader(repo_path, glob="**/*.py")
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=50,
        chunk_overlap=10
    )

    chunks = splitter.split_documents(docs)

    embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"), model="text-embedding-3-small")

    vector_db = FAISS.from_documents(chunks, embeddings)

    return vector_db



In [21]:
# from github_loader import clone_repo
# from rag_pipeline import build_vector_db
# from crew_setup import crew

# repo_url = input("Enter GitHub Repo URL: ")

repo_path = clone_repo(repo)

vector_db = build_vector_db(repo_path)
retriever = vector_db.as_retriever()


In [26]:
from crewai.tools import tool


@tool
def search_codebase(question: str):
    """Searches the repository for relevant code snippets based on a query."""
    docs = retriever.invoke(question)
    
    # Combine the results into a single string for the Agent to read
    return "\n\n".join([doc.page_content for doc in docs])

In [27]:
from crewai import Agent,Task,Crew
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini",api_key=os.getenv("OPENAI_API_KEY"))


# 2. Agent Definitions
code_reader = Agent(
    role="Lead Code Researcher",
    goal="Deep-dive into repository files to map out logic and data flow.",
    backstory="You are a veteran developer with a knack for reading complex codebases. You excel at identifying entry points and core modules.",
    verbose=True,
    tools=[search_codebase], # The Reader can call the search tool to get code context
    allow_delegation=False,
    llm=llm
)

bug_finder = Agent(
    role="Senior QA & Security Auditor",
    goal="Identify logic flaws, security vulnerabilities, and PEP8 violations.",
    backstory="You have a 'hacker mindset' focused on finding where code might break, especially regarding API integrations and data validation.",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

architect = Agent(
    role="Principal Software Architect",
    goal="Deconstruct the system design into high-level components and patterns.",
    backstory="You specialize in distributed systems and agentic workflows. You explain 'why' a system is built a certain way.",
    verbose=True,
    allow_delegation=True, # The Architect can ask the Reader for more info
    llm=llm
)

doc_writer = Agent(
    role="Senior Technical Writer",
    goal="Synthesize technical findings into clear, user-friendly documentation.",
    backstory="You bridge the gap between complex code and human understanding, producing world-class READMEs and API docs.",
    verbose=True,
    llm=llm
)

# 3. Task Definitions (Linked Context)
# Note how 'context' is used to pass findings from one agent to the next.


read_code_task = Task(
    description="""
Analyze the Travel Planner repository.

Tasks:
1. Identify the main script.
2. Identify helper modules.
3. Explain how user preferences are handled.

Then answer the user's question about the repository.

Question:
{user_question}

Use the search_codebase tool to retrieve relevant code from the vector database.
""",
    expected_output="""
A detailed map of the file structure and a summary of functional logic.
""",
    agent=code_reader
)

explain_arch_task = Task(
    description="Review the code structure and explain the architecture (e.g., RAG-based, Sequential, or Hierarchical).",
    expected_output="A technical architecture report including data flow diagrams in Mermaid.js format.",
    agent=architect,
    context=[read_code_task] # Architect needs the Reader's output
)

find_bugs_task = Task(
    description="Scan the logic for bugs, specifically looking at how the travel APIs are called and error handling.",
    expected_output="A list of critical bugs, warnings, and suggested refactors.",
    agent=bug_finder,
    context=[read_code_task]
)

generate_docs_task = Task(
    description="Create a comprehensive README.md. Include 'Getting Started', 'Architecture', and 'Known Issues' sections.",
    expected_output="A professional Markdown file ready for GitHub.",
    agent=doc_writer,
    context=[read_code_task, explain_arch_task, find_bugs_task] # Synthesis of all reports
)


In [28]:
from crewai import Process,Crew
from crewai import Crew
# from agents import code_reader, bug_finder, doc_writer, architect
# from tasks import read_code_task, explain_arch_task, find_bugs_task, generate_docs_task


# 4. The Crew (The Engine)
code_analysis_crew = Crew(
    agents=[code_reader, architect, bug_finder, doc_writer],
    tasks=[read_code_task, explain_arch_task, find_bugs_task, generate_docs_task],
    process=Process.sequential,
    verbose=True
)

In [ ]:
# 5. Kickoff
print("### Starting Codebase Analysis ###")

result = code_analysis_crew.kickoff(
    inputs={"user_question": "Where are DayItinerary objects created?"}
)

print("\n\n########################")
print("## FINAL ANALYSIS RESULT ##")
print("########################\n")
print(result)


### Starting Codebase Analysis ###


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9bf5bc37-78f1-486d-9bd4-b0ce85e67844                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Analyze the Travel Planner repository.                                                                         │
│                                                                                                                 │
│  Tasks:                                                                                                         │
│  1. Identify the main script.                                                                                   │
│  2. Identify helper modules.                                                                                    │
│  3. Explain how user preferences are handled.                                                                   │
│                                                                                                                 │
│  Then answer the user's question about the repository.                                                          │
│                                                                                                                 │
│  Question:                                                                                                      │
│  Where are DayItinerary objects created?                                                                        │
│                                                                                                                 │
│  Use the search_codebase tool to retrieve relevant code from the vector database.                               │
│                                                                                                                 │
│  ID: 2de3d5a2-e496-4cbf-adac-c42e9ed2d78d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Code Researcher                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Analyze the Travel Planner repository.                                                                         │
│                                                                                                                 │
│  Tasks:                                                                                                         │
│  1. Identify the main script.                                                                                   │
│  2. Identify helper modules.                                                                                    │
│  3. Explain how user preferences are handled.                                                                   │
│                                                                                                                 │
│  Then answer the user's question about the repository.                                                          │
│                                                                                                                 │
│  Question:                                                                                                      │
│  Where are DayItinerary objects created?                                                                        │
│                                                                                                                 │
│  Use the search_codebase tool to retrieve relevant code from the vector database.                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#13) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_codebase                                                                                          │
│  Args: {'question': 'main script of the Travel Planner repository'}                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9bf5bc37-78f1-486d-9bd4-b0ce85e67844                                                                       │
│  Final Output: ```markdown                                                                                      │
│  # Travel Planner Repository                                                                                    │
│                                                                                                                 │
│  Welcome to the Travel Planner repository! This application is designed to help users organize their travel     │
│  itineraries by taking user preferences into account and generating structured daily plans.                     │
│                                                                                                                 │
│  ## Getting Started                                                                                             │
│                                                                                                                 │
│  To set up the Travel Planner on your local machine, follow these steps:                                        │
│                                                                                                                 │
│  1. **Clone the Repository**                                                                                    │
│     ```bash                                                                                                     │
│     git clone https://github.com/yourusername/travel-planner.git                                                │
│     cd travel-planner                                                                                           │
│     ```                                                                                                         │
│                                                                                                                 │
│  2. **Install Dependencies**                                                                                    │
│     Ensure you have Python 3.x installed. Then, install required packages using:                                │
│     ```bash                                                                                                     │
│     pip install -r requirements.txt                                                                             │
│     ```                                                                                                         │
│                                                                                                                 │
│  3. **Run the Application**                                                                                     │
│     Start the application by running the main script:                                                           │
│     ```bash                                                                                                     │
│     python app.py                                                                                               │
│     ```                                                                                                         │
│                                                                                                                 │
│  4. **Input User Preferences**                                                                                  │
│     Follow the prompts to enter your travel preference

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯